# Notebook 6: End-to-End System Demo
## SmartCache ML: Service Integration, Audit Logs & Deployment Guide

### 1. Problem Overview
In this final capstone notebook, we connect all modular components into the unified **SmartCacheService**:
1. Persistent Semantic Cache with SQLite.
2. Query Complexity ML Router.
3. Interchangeable LLM Provider (MockProvider / local Ollama).
4. Real-time cost, latency, and hit-rate logging.
5. Exporting benchmark artifacts for reports and dashboards.

We also demonstrate how to launch the FastAPI REST API and the Streamlit monitoring dashboard.

---
### Learning Objectives
By the end of this notebook, you will be able to:
- Orchestrate caching, routing, inference, and telemetry in a single clean Python service.
- Replay and log production traffic traces.
- Export benchmark results for stakeholder reporting.
- Run and test the FastAPI backend and Streamlit dashboard locally.

In [ ]:
import pandas as pd
from smartcache.config import settings
from smartcache.service import SmartCacheService

# Initialize end-to-end service
service = SmartCacheService()
# Reset cache for a fresh demonstration
service.cache.invalidate()
service.cost_tracker.reset()

print("SmartCacheService ready!")
print("Provider:", settings.llm_provider)
print("Embedding model:", settings.embedding_model_name)

### 2. Replaying Production Traffic Traces
Let's simulate a sequence of diverse real-world requests and observe the service decisions.

In [ ]:
traffic_queries = [
    # 1. First-time factual query (Expect: Cache Miss -> Router 'small' -> Stored in Cache)
    "What is the capital of France?",
    
    # 2. Immediate paraphrase (Expect: Cache Hit! ~0.0 cost)
    "Can you tell me the capital city of France?",
    
    # 3. Complex reasoning query (Expect: Cache Miss -> Router 'large' -> Stored in Cache)
    "Explain how QuickSort partitions elements and its worst-case algorithmic complexity.",
    
    # 4. Paraphrase of complex query (Expect: Cache Hit! ~0.0 cost)
    "How does quicksort partition work and what is its big-O time complexity?",
    
    # 5. Adversarial near-duplicate (Expect: Cache Miss -> correctly rejected from France capital cache)
    "What is the largest port city in France?",
    
    # 6. Real-time volatile query (Expect: Cache Miss -> Router 'small' -> NOT cached)
    "What is the current date and time right now?"
]

records = []
for q in traffic_queries:
    is_cacheable = "right now" not in q.lower()
    res = service.process_query(q, is_cacheable=is_cacheable)
    records.append({
        "Query": res.query[:45] + "...",
        "Cache Hit": res.cache_hit,
        "Sim Score": f"{res.similarity_score:.3f}" if res.similarity_score else "N/A",
        "Route": res.selected_route,
        "Latency (ms)": res.latency_ms,
        "Cost ($)": f"${res.estimated_cost_usd:.6f}",
        "Reason": res.routing_reason[:45] + "..."
    })

trace_df = pd.DataFrame(records)
trace_df

### 3. Reviewing Aggregate System Metrics

In [ ]:
summary = service.cost_tracker.get_summary()
print("=== END-TO-END SYSTEM SUMMARY ===")
print(f"Total Requests:         {summary['total_requests']}")
print(f"Cache Hits:             {summary['cache_hits']}")
print(f"Cache Misses:           {summary['cache_misses']}")
print(f"Cache Hit Rate:         {summary['cache_hit_rate'] * 100:.1f}%")
print(f"Total Estimated Cost:   ${summary['total_cost_usd']:.6f}")
print(f"Mean Latency:           {summary['mean_latency_ms']:.2f} ms")
print(f"P50 Latency:            {summary['p50_latency_ms']:.2f} ms")
print(f"P95 Latency:            {summary['p95_latency_ms']:.2f} ms")
print(f"Route Distribution:     {summary['route_distribution']}")

# Export benchmark summary
report_path = settings.project_root / "reports" / "benchmark_summary.csv"
trace_df.to_csv(report_path, index=False)
print(f"\nSaved execution trace to {report_path}")

### 4. Running the REST API and Streamlit Dashboard
To launch the complete interactive application:

1. **FastAPI REST API**:
   ```bash
   uvicorn api.main:app --host 127.0.0.1 --port 8000 --reload
   ```
   Interactive OpenAPI docs available at `http://127.0.0.1:8000/docs`.

2. **Streamlit Monitoring Dashboard**:
   ```bash
   streamlit run dashboard/app.py
   ```
   Opens browser dashboard displaying real-time metrics, cache statistics, and policy comparisons.

---
### 5. Architectural Limitations & Production Next Steps
In an engineering interview, be honest about system limitations:
1. **SQLite Concurrency**: SQLite locks the database file during writes. For high-concurrency production deployments (> 1,000 req/sec), replace SQLite with PostgreSQL (pgvector) or Redis.
2. **Flat NumPy Search vs Hierarchical Indexes**: Flat dot product is exact and fast for < 50,000 entries. For millions of embeddings, introduce an approximate nearest neighbor (ANN) index such as HNSW or ScaNN.
3. **Embedding Drift**: Changes in the underlying embedding model require re-indexing existing cache entries.
4. **Cache Invalidation on Real-world Updates**: Semantic caches require careful invalidation when authoritative source documents change.